# Planner

In [1]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(ROOT_DIR)

c:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2


In [2]:
from __future__ import annotations
import math
from datetime import datetime, timedelta
from app.schemas import (
    PlanningRequest, POI, StructuredIntent, Itinerary, DayPlan, ItineraryStop, ItineraryScore
)
from app.graph import (
    haversine_km, cluster_pois, group_by_cluster, score_all_pois, build_knn_graph
)
from app.config import settings
from app.utils.save import save_artifact
from app.providers.provider import run_retrieval

In [3]:
from app.extractor import extract_intent
from app.schemas import StructuredIntent
import json

# request = PlanningRequest(
#     query="5-day Tokyo trip, interested in museums and food, moderate budget, staying near Shinjuku, avoid excessive walking"
# )

# intent = await extract_intent(request.query)
# save_artifact('123456', 'intent', intent)
from app.schemas import (
    StructuredIntent,
    Preferences,
    Constraints,
)

intent = StructuredIntent(
    destination="Tokyo",
    days=5,
    stay_location="Shinjuku",
    is_international=True,
    budget="medium",
    preferences=Preferences(
        museums=0.8,
        food=0.8,
        nightlife=0.0,
        nature=0.0,
        shopping=0.0,
        arts=0.0,
        history=0.0,
        wellness=0.0,
    ),
    constraints=Constraints(
        walking_limit_km=5.5,
        must_visit=[],
        avoid=[],
        budget_per_day_usd=None,
    ),
)

pois_GA, lat, lon = await run_retrieval(
    source="GA",
    intent=intent,
    debug=True
)

pois_FS, lat, lon = await run_retrieval(
    source="FS",
    intent=intent,
    debug=True
)
pois = pois_GA + pois_FS
save_artifact('123456', 'POIS', pois)


=== RETRIEVAL START ===
Provider: GA
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 847
Output POIs: 755
Dropped: 92
By Category: {'museums': 3, 'food': 89}

=== AFTER DEDUP ===
Total POIs: 755
{'museums': 352, 'food': 403}

=== MUST VISIT FILTER ===
Must Visit Targets: []
Matched POIs: 0

=== AVOID FILTER ===
Avoid Categories: set()
Removed: 0

=== FINAL RESULT ===
Total POIs: 755
Must Visit POIs: 0
Regular POIs: 755
{'museums': 352, 'food': 403}


=== RETRIEVAL START ===
Provider: FS
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 98
Output POIs: 49
Dropped: 49


In [5]:
pois

[POI(id='GA_5103a3810e2976614059affc43447bd84140f00103f901ea60af540000000092031be5b9b3e5928ce7a588e5bfb5e5b195e7a4bae8b387e69699e9a4a8', name='平和祈念展示資料館', lat=35.6912618, lon=139.6925118, category='museums', tags=['entertainment', 'entertainment.museum'], popularity_score=0.5, opening_hours={}, avg_duration_minutes=60, estimated_cost_usd=0.0, rating=3.5, address='Memorial Museum for Soldiers, Detainees in Siberia, and Postwar Repatriates, Tocho-dori Ave., Nishi-Shinjuku, Shinjuku, Nishi-Shinjuku 2 160-0023, Japan', source='geoapify', utility_score=0.72, is_anchor=False),
 POI(id='GA_51938c9c853d766140593d433866d9d74140f00103f901c264af540000000092031be69687e58c96e5ada6e59c92e69c8de9a3bee58d9ae789a9e9a4a8', name='文化学園服飾博物館', lat=35.686322, lon=139.69501, category='museums', tags=['entertainment', 'entertainment.museum'], popularity_score=0.5, opening_hours={}, avg_duration_minutes=60, estimated_cost_usd=0.0, rating=3.5, address='Bunka Gakuen Costume Museum, 公衆電話, Nishi-Shinjuku, Shinjuku

In [4]:
from app.graph import score_all_pois
import json

scored_pois = score_all_pois(pois, intent)

print(
    json.dumps(
        [p.model_dump() for p in scored_pois],
        indent=2,
        ensure_ascii=False
    )
)

[
  {
    "id": "GA_5103a3810e2976614059affc43447bd84140f00103f901ea60af540000000092031be5b9b3e5928ce7a588e5bfb5e5b195e7a4bae8b387e69699e9a4a8",
    "name": "平和祈念展示資料館",
    "lat": 35.6912618,
    "lon": 139.6925118,
    "category": "museums",
    "tags": [
      "entertainment",
      "entertainment.museum"
    ],
    "popularity_score": 0.5,
    "opening_hours": {},
    "avg_duration_minutes": 60,
    "estimated_cost_usd": 0.0,
    "rating": 3.5,
    "address": "Memorial Museum for Soldiers, Detainees in Siberia, and Postwar Repatriates, Tocho-dori Ave., Nishi-Shinjuku, Shinjuku, Nishi-Shinjuku 2 160-0023, Japan",
    "source": "geoapify",
    "utility_score": 0.72,
    "is_anchor": false
  },
  {
    "id": "GA_51938c9c853d766140593d433866d9d74140f00103f901c264af540000000092031be69687e58c96e5ada6e59c92e69c8de9a3bee58d9ae789a9e9a4a8",
    "name": "文化学園服飾博物館",
    "lat": 35.686322,
    "lon": 139.69501,
    "category": "museums",
    "tags": [
      "entertainment",
      "entertainmen

#### Graph:

- utility score
- builds KNN using haversine
- cluster_pois uses DBSCAN
- group_by_cluster


Updates:

- pre ML filter for bad POIs
- train global algo for geospatial clustering
- use h3 indexes for better results
- benchmark the following: KNN, DBSCAN, HDBSCAN
- DS: OSM

Pre filter:

1. Filter bad names, unless in "must"/in constraint/preference
2. Whether POI in both geoapify & FS - do not automatically reject, up vote it
3. Whether POI has external links 
4. Bayesian rating & review confidence
5. Category rating

In [21]:
from pydantic import BaseModel
from rapidfuzz import fuzz
import unicodedata

class QualityScore(BaseModel):
    id: str
    name_score: float = 1.0
    source_score: float = 0.0
    bayesian_rating: float = 0.0
    review_confidence: float = 0.0
    category_score: float = 0.0
    external_link_score: float = 0.0
    quality_score: float = 0.0
    overall_score: float = 0.0
    reasons: list[str] = []

    
BAD_TERMS = {
    "atm",
    "parking",
    "bus stop",
    "toilet",
    "restroom",
    "taxi stand",
    "charging station"
}
def normalize_name(name: str) -> str:
    return unicodedata.normalize("NFKC", name).lower().strip()

class Filter:
    def __init__(self):
        pass
    
    @staticmethod
    def _is_same_poi(name1: str, name2: str) -> bool:
        n1 = normalize_name(name1)
        n2 = normalize_name(name2)

        score = fuzz.token_sort_ratio(n1, n2)

        return score >= 90

    @staticmethod
    def _score_downBT(name: str, intent) -> float:
        name_lower = name.lower()
        must_visit = {p.lower() for p in intent.constraints.must_visit}
        if any(mv in name_lower or name_lower in mv for mv in must_visit): 
            return 1.0
        for term in BAD_TERMS:
            if term in name_lower:
                return 0.0
        return 1.0
    
    def _score_source(self, poi, pois):
        for other in pois:
            if other.id == poi.id:
                continue
            if self._is_same_poi(other.name, poi.name):
                return 1.0
        return 0.0

    # main class
    def score_filter(self, pois) -> list[QualityScore]:
        poi_scores = []
        for poi in pois:
            score_BT = self._score_downBT(poi.name, intent)
            score_source = self._score_source(poi, pois)

            poi_scores.append(
                QualityScore(
                    id = poi.id,
                    name_score=score_BT,
                    source_score=score_source,

                )
            )

        return poi_scores



        

    

filter_obj = Filter()
poi_scores = filter_obj.score_filter(pois)

In [22]:
has_multi_source = any(
    score.source_score == 1
    for score in poi_scores
)

In [23]:
has_multi_source

True

In [24]:
count = sum(
    1
    for score in poi_scores
    if score.source_score == 1
)

In [25]:
count

10